# Understanding the Juno-60 Chorus

Todo: write article, start with history, move into BBD explanation, put all together into chorus. add interactive elements (upload file or record audio to demonstrate chorus) and also animations.


Resources used:

- Helpful repo that provides an analysis on the Juno-60 chorus: https://github.com/pendragon-andyh/Juno60/tree/master/Chorus
- WebAudio API emulation of the classic Roland Juno-60 synthesizer: https://github.com/pendragon-andyh/junox
    - Demo: https://pendragon-andyh.github.io/junox/demo/juno60-nexusUI.html
- A collaborative reverse-engineering discussion using a "gray-box" method: https://github.com/jpcima/rc-effect-playground/issues/2
- Comparative reference covering modulation rates, basic delay times, and BBD chip variations across all Roland pre-digital chorus circuits: https://www.florian-anwander.de/roland_string_choruses/
- Thread discussing key LFO parameters from the official manuals: https://www.kvraudio.com/forum/viewtopic.php?t=489346
- A hands-on teardown and standalone build using a real 106 chorus board. Explains the control signals (I/II mode logic, HPF switching, VCA): https://hkadesign.org.uk/106chorus.html
- A brief history of the Juno Synth line: https://www.perfectcircuit.com/signal/roland-juno-history
- Video explaining Bucket Brigade Delay (BBD): https://www.youtube.com/watch?v=PD48jm51K6g
- An (Excellent) history of chorus: https://www.youtube.com/watch?v=k2_LQaORseY

Relevant VSTs:
- A free Juno-60 chorus emulation from TAL Software: https://tal-software.com/products/tal-chorus-lx
- A paid Juno-60 chorus plugin by Roland: https://www.roland.com/us/products/rc_juno-60_chorus/

# Article

#### What is Chorus?
The goal of chorus is to make a duplicated, detuned signal.
- The psychoacoustic illusion: why slightly detuned copies sound "thicker"
- The three ingredients: pitch deviation, time offset, and mix ratio
- Chorus vs. flange vs. vibrato — where the borders are

#### The History of Chorus

##### Choirs and String Sections
- Natural ensemble detuning — why 16 violinists sound bigger than one
- Early tape doubling tricks (ADT at Abbey Road, ca. 1966)

##### Rotating Speakers — The Leslie Cabinet
- Doppler shift as modulated pitch: the first mechanical chorus
- Horn vs. woofer: different speeds, different modulation depths

##### Tape and Early Analog Electronics
- Tape flanging and the invention of the effect name
- Boss CE-1 (1976): the first dedicated chorus pedal, spawning a lineage
- Roland's string ensemble chips and the Juno-6 debut (1982)

#### Effect Mathematics

##### The Modulated Delay Line
- `y(t) = x(t) + x(t − d(t))` where `d(t) = d₀ + A·sin(2πft)`
- Why changing delay time equals changing pitch (instantaneous frequency)
- Chorus vs. flanger delay ranges: ~1–30ms vs. 0–15ms

##### Stereo Image from a Mono Source
- L and R LFOs 180° out of phase — the Juno trick
- Wet/dry mix ratio and its effect on perceived width
- Comb filtering artifacts and how to minimise them

##### LFO Shapes and Their Sonic Character
- Triangle (Juno I/II) vs. sine (I+II mode) vs. random for organic feel
- Linear interpolation in the delay buffer — why it matters for smooth pitch

#### Bucket Brigade Delay (BBD)

##### How a BBD Chip Works
- Discrete capacitor stages clocked in sequence — an analog shift register
- MN3009: 256 stages, 0.64–12.8ms range, ~70kHz effective sample rate
- Anti-aliasing LPF before, reconstruction LPF after — why both are needed

##### BBD Artifacts and Why We Love Them
- Clock noise, charge-well saturation, and the "noise floor shimmer"
- No compander on the Juno (unlike the Dimension-D) — distortion is the character
- Emulating BBD noise in DSP: shaped noise floor, subtle clock bleed

#### DSP Implementation

##### The Circular Delay Buffer
- Ring buffer setup: write pointer, fractional read pointer
- **Code:** initialise buffer, advance write head, compute read position

##### Linear Interpolation (and Why You Need It)
- Fractional sample reads without interpolation = audible zipper noise
- **Code:** lerp between floor/ceil samples; Hermite spline as a step-up option

##### The LFO
- Accumulator-based triangle oscillator in samples, not seconds
- Phase offset for right channel: `lfo_r = -lfo_l` (the 180° inversion)
- **Code:** lfo tick, depth scaling to milliseconds, converting ms → samples

##### Pre- and Post-BBD Filtering
- Simple 1-pole (6dB/oct) LPF before the delay line — cutoff ~8kHz
- Reconstruction LPF after — matching the analog rolloff character
- **Code:** one-pole filter struct, coefficient calculation from cutoff Hz

##### Wet/Dry Mixing and Output Stage
- Juno ratio: processed signal at −1.62 dB slightly louder than dry
- **Code:** complete per-sample `process()` function, stereo output struct

#### Putting It Together — Juno Chorus in Your Synth

##### Mode I, II, and I+II
- Switching LFO rate and depth per mode (table of exact values)
- Mode I+II: sine LFO at ~9.75 Hz, much reduced depth — vibrato character
- **Code:** mode enum, param struct, `chorus_set_mode()`

##### Integration into the Synth Signal Chain
- Placement: after HPF, before output — why order matters
- Sample rate considerations: delay buffer sizing at 44.1k vs. 48k
- **Code:** `chorus_process(chorus_t*, float in) → stereo_sample_t`

#### Tuning by Ear
- Using a pulse wave test signal to visualise delay offset in a DAW
- Checking L/R phase inversion: the channels should mirror, not match
- Common mistakes: no interpolation, wrong LFO polarity, missing pre-filter

#### Going Further
- BBD noise emulation: shaped noise added post-delay for analog warmth
- Per-voice chorus, stereo input variant, CPU cost tradeoffs



#### Key Parameters (summary from all sources)

| Parameter | Mode I | Mode II | Mode I+II |
|---|---|---|---|
| LFO rate | ~0.5 Hz triangle | ~0.83 Hz triangle | ~8–10 Hz sine |
| LFO depth | high | high | much lower |
| L/R phase | 180° inverted | 180° inverted | 180° inverted |
| BBD stages | 256 (MN3009) | 256 (MN3009) | 256 (MN3009) |
| Pre-BBD filter | 12dB LPF | 12dB LPF | 12dB LPF |
| Wet/dry mix | ~−1.62dB (processed slightly louder) | same | same |

The chorus notably has **no compander** (unlike the Dimension-D), so the BBD noise is part of the sound.